# Variant 04. Restaurant Performance Analysis

**Domain:** Restaurant Performance, Kyrgyzstan. All money is in KGS. Data are synthetic.
Tools used: NumPy, Pandas, Matplotlib only. The original CSV files are never modified; all cleaning is done on a copy.

## TASK 1. Initial Data Exploration

Both files start with a BOM character, so they are read with `encoding="utf-8-sig"`. Otherwise the first column would be called `\ufeffTip Amount`.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

raw = pd.read_csv("04_restaurant_raw.csv", encoding="utf-8-sig")
lookup = pd.read_csv("04_restaurant_lookup.csv", encoding="utf-8-sig")
print("raw shape:", raw.shape, "| lookup shape:", lookup.shape)

raw shape: (840, 12) | lookup shape: (7, 3)


**1.1 Shape.** The main table has 840 rows and 12 columns; the lookup table has 7 rows and 3 columns.

**1.2 First 10 rows:**

In [2]:
raw.head(10)

,Tip Amount,Payment Method,Customer Count,Waiter,Unit Price,Quantity,dish_name_raw,Dish Category,Meal Type,Order Date,Branch,Order ID
0,No,Basic,No,Premium,19094,38,Yes,Category A,Standard,2025-06-15,Bishkek,04_-00000
1,Yes,Basic,No,Premium,41266,21,Yes,Category B,Premium,28.11.2024,Bishkek,04_-00001
2,No,Basic,No,Premium,48267,16,Yes,Category A,Premium,2024-12-08,Bishkek,04_-00002
3,Yes,Premium,Yes,Basic,25182,49,No,Category C,Basic,2024-02-09,Tokmok,04_-00003
4,No,Basic,Yes,Standard,22174,39,No,Category C,Basic,23.12.2025,Bishkek,04_-00004
5,Yes,Premium,Yes,Premium,30596,41,No,Category A,Basic,16.05.2024,Bishkek,04_-00005
6,Yes,Standard,No,Standard,4950,14,No,Category C,Basic,2025-10-14,Bishkek,04_-00006
7,No,Standard,No,Premium,39254,4,Yes,Category A,Premium,04.04.2025,Jalal-Abad,04_-00007
8,No,Standard,No,Basic,16132,25,Yes,Category A,Basic,2025-09-14,Karakol,04_-00008
9,Yes,Basic,No,Basic,49238,11,Yes,Category C,Basic,27.06.2024,Tokmok,04_-00009


**1.3 Column names and types (`.info()`).** Column names mix spaces, capitals and snake_case (`dish_name_raw`). `Order Date` is `object` (text), not a date. `Unit Price` and `Quantity` are already `int64` in this file, although the task describes `Quantity` as text; the `astype()` conversion is still applied in Task 2 so the notebook is safe if the text version is used.

In [3]:
raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 840 entries, 0 to 839
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   Tip Amount      840 non-null    str  
 1   Payment Method  840 non-null    str  
 2   Customer Count  840 non-null    str  
 3   Waiter          840 non-null    str  
 4   Unit Price      840 non-null    int64
 5   Quantity        840 non-null    int64
 6   dish_name_raw   840 non-null    str  
 7   Dish Category   825 non-null    str  
 8   Meal Type       827 non-null    str  
 9   Order Date      840 non-null    str  
 10  Branch          840 non-null    str  
 11  Order ID        840 non-null    str  
dtypes: int64(2), str(10)
memory usage: 78.9 KB


**1.4 Unique values.** `Meal Type` has 11 spellings for 3 real categories (`Basic`, `basic`, ` Basic `, ...) plus missing values. `Payment Method` is clean: only `Basic`, `Premium`, `Standard`.

In [4]:
print(raw["Meal Type"].unique())
print(raw["Meal Type"].value_counts(dropna=False))
print(raw["Payment Method"].unique())
print(raw["Payment Method"].value_counts(dropna=False))

<StringArray>
[  'Standard',    'Premium',      'Basic',    'premium',          nan,
   'standard',    ' Basic ',  ' Premium ', ' Standard ',      'basic',
 ' standard ']
Length: 11, dtype: str
Meal Type
Premium       269
Basic         258
Standard      245
premium        13
NaN            13
standard       10
 Basic          8
 Standard       8
basic           8
 Premium        7
 standard       1
Name: count, dtype: int64
<StringArray>
['Basic', 'Premium', 'Standard']
Length: 3, dtype: str
Payment Method
Standard    292
Premium     275
Basic       273
Name: count, dtype: int64


**1.5 Missing values.** `Meal Type` has 13 missing and `Dish Category` has 15 missing. No other column has missing values (the "N/A" mentioned in the task notes does not occur in this file).

In [5]:
raw.isna().sum()

Tip Amount         0
Payment Method     0
Customer Count     0
Waiter             0
Unit Price         0
Quantity           0
dish_name_raw      0
Dish Category     15
Meal Type         13
Order Date         0
Branch             0
Order ID           0
dtype: int64

**1.6 Duplicates.** There are 19 fully identical rows (the same `Order ID` and all other fields), shown below. There are 20 repeated `Order ID`s, so one `Order ID` is reused by two *different* records. That one is not a copy and is kept.

In [6]:
print("duplicate rows:", raw.duplicated().sum())
print("duplicate Order IDs:", raw["Order ID"].duplicated().sum())
raw[raw.duplicated(keep=False)].sort_values("Order ID").head(6)

duplicate rows: 19
duplicate Order IDs: 20


,Tip Amount,Payment Method,Customer Count,Waiter,Unit Price,Quantity,dish_name_raw,Dish Category,Meal Type,Order Date,Branch,Order ID
28,Yes,Basic,Yes,Premium,19088,13,No,Category B,Premium,12.04.2025,Jalal-Abad,04_-00028
837,Yes,Basic,Yes,Premium,19088,13,No,Category B,Premium,12.04.2025,Jalal-Abad,04_-00028
831,No,Basic,No,Premium,32078,45,No,Category B,Standard,2025-05-31,Jalal-Abad,04_-00228
228,No,Basic,No,Premium,32078,45,No,Category B,Standard,2025-05-31,Jalal-Abad,04_-00228
835,Yes,Standard,No,Premium,4758,22,Yes,Category D,Premium,21.02.2024,Karakol,04_-00236
236,Yes,Standard,No,Premium,4758,22,Yes,Category D,Premium,21.02.2024,Karakol,04_-00236


**1.7 Variable types and data quality problems.**

- **Numerical:** `Unit Price`, `Quantity`.
- **Categorical:** `Order ID`, `Order Date` (text), `Branch`, `Meal Type`, `Dish Category`, `Payment Method`, `Waiter`, plus `Tip Amount`, `Customer Count`, `dish_name_raw`.

Problems found:

1. `Meal Type` inconsistent: `premium`, ` Basic `, ` standard ` (case and spaces). 13 values missing.
2. `Dish Category` has 15 missing values.
3. 19 exact duplicate rows.
4. `Unit Price` has extreme outliers: the max is 1,728,760 while the median is 26,475 (about 65 times larger).
5. `Order Date` is text in two formats: `2025-06-15` and `28.11.2024` (415 of 840 rows use dots).
6. Column names are inconvenient (spaces, capitals, mixed style) and the order is illogical (the ID is last).
7. **Misleading columns:** `Tip Amount`, `Customer Count` and `dish_name_raw` contain only `Yes`/`No`. They are not amounts or counts. `Payment Method` and `Waiter` contain `Basic/Premium/Standard`, not real payment types or names. This cannot be fixed; it affects Task 3.3 (see there).
8. The lookup table has duplicate keys that differ only by case (`Basic` and `basic`, with different segments) and one row with an empty key.

## TASK 2. Data Cleaning and Preparation

All cleaning is done on the copy `df`; `raw` stays untouched.

**2.1 Missing values.** I cannot know the real `Meal Type` or `Dish Category` of these orders. Guessing (mode fill) would invent data and would add to the biggest category. Dropping rows would throw away valid prices and quantities. So the missing values are labelled `Unknown`: the rows stay, and the gap stays visible.

In [7]:
df = raw.copy()
df["Meal Type"] = df["Meal Type"].fillna("Unknown")
df["Dish Category"] = df["Dish Category"].fillna("Unknown")
print(df[["Meal Type", "Dish Category"]].isna().sum().to_dict())
print((df["Meal Type"] == "Unknown").sum(), (df["Dish Category"] == "Unknown").sum())

{'Meal Type': 0, 'Dish Category': 0}
13 15


**2.2 Standardise `Meal Type`.** `.str.strip()` removes spaces and `.str.title()` unifies case. Mapping: `basic`, ` Basic ` → `Basic`; `premium`, ` Premium ` → `Premium`; `standard`, ` standard `, ` Standard ` → `Standard`. Result: 3 categories + `Unknown`.

In [8]:
df["Meal Type"] = df["Meal Type"].str.strip().str.title()
mapping = {"Basic": "basic / ' Basic '", "Premium": "premium / ' Premium '", "Standard": "standard / ' standard ' / ' Standard '"}
print(df["Meal Type"].value_counts())

Meal Type
Premium     289
Basic       274
Standard    264
Unknown      13
Name: count, dtype: int64


**2.3 Duplicates.** `drop_duplicates()` is run *after* standardising so that rows that differ only by spelling are also caught. It removed 19 rows (840 → 821). One `Order ID` is still repeated, but with different content, so it is a different record and stays.

In [9]:
before = len(df)
df = df.drop_duplicates()
print("removed:", before - len(df), "| rows now:", len(df))
print("Order IDs still repeated:", df["Order ID"].duplicated().sum())

removed: 19 | rows now: 821
Order IDs still repeated: 1


**2.4 Outliers in `Unit Price`.** I use the IQR rule (covered in class): a value is an outlier if it is below Q1 − 1.5·IQR or above Q3 + 1.5·IQR. Q1 = 13,191 and Q3 = 38,396, so the upper bound is 76,203.5. **10 values** are above it (107,600 up to 1,728,760); the lower bound is negative, so no low outliers are flagged. All other prices are below 50,000, so the 10 are clearly separated from the data. I remove them instead of capping, because I cannot know the real price (it looks like an entry error, e.g. extra digits) and a capped value would be invented.

In [10]:
q1, q3 = df["Unit Price"].quantile([0.25, 0.75])
iqr = q3 - q1
low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
mask = (df["Unit Price"] < low) | (df["Unit Price"] > high)
print(f"Q1={q1}, Q3={q3}, IQR={iqr}, bounds=({low}, {high})")
print("outliers:", mask.sum())
df.loc[mask, ["Order ID", "Unit Price", "Quantity"]].sort_values("Unit Price")

Q1=13191.0, Q3=38396.0, IQR=25205.0, bounds=(-24616.5, 76203.5)
outliers: 10


,Order ID,Unit Price,Quantity
559,04_-00559,107600,39
545,04_-00545,364360,12
11,04_-00011,1114840,27
638,04_-00638,1257240,39
709,04_-00709,1317680,3
637,04_-00637,1424200,2
292,04_-00292,1508520,28
718,04_-00718,1520640,6
331,04_-00331,1635440,42
485,04_-00485,1728760,36


In [11]:
df = df[~mask].copy()
print("rows after removing outliers:", len(df), "| max price:", df["Unit Price"].max())

rows after removing outliers: 811 | max price: 49887


**2.5 `Quantity` to numeric** with `astype(int)`.

In [12]:
df["Quantity"] = df["Quantity"].astype(int)
print(df["Quantity"].dtype)

int64


**2.6 `Order Date` to datetime.** `pd.to_datetime` on a mixed column would guess the format and could swap day and month (`12.04.2025` is 12 April, not 4 December). So I split by format: values with `-` are `YYYY-MM-DD`, the others are `DD.MM.YYYY`. Each part is parsed with an explicit `format=`, then they are joined back by index. The range is 2024-01-01 to 2025-12-31.

In [13]:
iso = df["Order Date"].str.contains("-")
d1 = pd.to_datetime(df.loc[iso, "Order Date"], format="%Y-%m-%d")
d2 = pd.to_datetime(df.loc[~iso, "Order Date"], format="%d.%m.%Y")
df["Order Date"] = pd.concat([d1, d2]).sort_index()
print(df["Order Date"].dtype, df["Order Date"].min(), df["Order Date"].max())

datetime64[us] 2024-01-01 00:00:00 2025-12-31 00:00:00


**2.7 Column names and order.** Names are lowercase snake_case. `Unit Price` becomes `unit_price`. The order is logical: identifiers and time, then place, then product, then payment/people, then numbers.

In [14]:
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")
df = df[["order_id", "order_date", "branch", "meal_type", "dish_category", "dish_name_raw",
         "payment_method", "waiter", "customer_count", "quantity", "unit_price", "tip_amount"]]
df.columns.tolist()

['order_id',
 'order_date',
 'branch',
 'meal_type',
 'dish_category',
 'dish_name_raw',
 'payment_method',
 'waiter',
 'customer_count',
 'quantity',
 'unit_price',
 'tip_amount']

**2.8 Merge with the lookup.** The lookup has the same case problem as the main table: `Basic` and `basic` have different segments, and one row has an empty key. If I merged it as is, orders would be duplicated or wrongly matched. I keep only the properly spelled keys (`Basic`, `Premium`, `Standard`) and drop the lowercase duplicates and the empty-key row. I cannot tell which of the two conflicting segments is correct; the properly spelled key is the choice I can justify.

In [15]:
lk = lookup.copy()
lk.columns = lk.columns.str.strip().str.lower().str.replace(" ", "_")
lk = lk.dropna(subset=["meal_type"])
lk = lk[lk["meal_type"] == lk["meal_type"].str.title()]
print(lk)

  meal_type business_segment priority
0     Basic        Segment 1     High
1   Premium        Segment 2   Medium
2  Standard        Segment 3      Low


I use a **left merge**: every main record must stay. The `assert` confirms the row count did not change. **Unmatched: 12 rows**, all with `meal_type = Unknown`, which has no lookup entry; their `business_segment` and `priority` remain empty.

In [16]:
n = len(df)
df = df.merge(lk, on="meal_type", how="left")
assert len(df) == n
print("unmatched rows:", df["business_segment"].isna().sum())
print(df.loc[df["business_segment"].isna(), "meal_type"].value_counts())

unmatched rows: 12
meal_type
Unknown    12
Name: count, dtype: int64


**2.9 Final check and save.** The only remaining missing values are those 12 unmatched lookup fields. Types are correct. 811 rows and 14 columns are saved to `cleaned_data.csv` with `index=False`.

In [17]:
print(df.isna().sum()[lambda s: s > 0])
print(df.dtypes)
df.to_csv("cleaned_data.csv", index=False)
print(df.shape)

business_segment    12
priority            12
dtype: int64
order_id                       str
order_date          datetime64[us]
branch                         str
meal_type                      str
dish_category                  str
dish_name_raw                  str
payment_method                 str
waiter                         str
customer_count                 str
quantity                     int64
unit_price                   int64
tip_amount                     str
business_segment               str
priority                       str
dtype: object
(811, 14)


## TASK 3. NumPy Practical Application

### 3.1 Array creation and conversion

In [18]:
qty_arr = np.array(df["quantity"])
price_arr = np.array(df["unit_price"])
for name, a in [("qty_arr", qty_arr), ("price_arr", price_arr)]:
    print(name, a.shape, a.ndim, a.size, a.dtype)
coef = np.linspace(0.8, 1.2, 6)
print("coef", coef, coef.shape, coef.ndim, coef.size, coef.dtype)

qty_arr (811,) 1 811 int64
price_arr (811,) 1 811 int64
coef [0.8  0.88 0.96 1.04 1.12 1.2 ] (6,) 1 6 float64


Both arrays have 811 elements, one dimension, shape `(811,)` and dtype `int64`. `coef` is `float64` with 6 values from 0.8 to 1.2 (step 0.08).

**List vs array.** A Python list can hold items of different types and every item is a separate object. A NumPy array holds one type in one continuous block of memory, supports whole-array maths and is much faster.

### 3.2 Indexing and slicing

In [19]:
print(price_arr[:8])
print(price_arr[::2][:8], len(price_arr[::2]))

[19094 41266 48267 25182 22174 30596  4950 39254]
[19094 48267 22174  4950 16132 39073 26416 45193] 406


`arr[:8]` is a slice: it returns a sub-array with the first 8 elements. `arr[::2]` is a slice with step 2: it returns every second element (positions 0, 2, 4, ...), 406 values here. Indexing (`arr[3]`) returns one single element; slicing returns an array (a view of the same memory, not a copy).

### 3.3 Vectorization

`Tip Amount` contains `Yes`/`No`, not money, so it cannot be subtracted as an amount. I encode it as a flag (`Yes` = 1, `No` = 0) and apply the required formula unchanged: `net = qty_arr * price_arr - extra_arr`. The tip term is therefore at most 1 KGS, which is negligible, and the result is essentially the **transaction value** (Quantity × Unit Price). This is a limitation of the data, not of the method.

In [20]:
extra_arr = (df["tip_amount"] == "Yes").astype(int).to_numpy()
net = qty_arr * price_arr - extra_arr
print(net[:5], net.dtype)
print("total net:", net.sum())

[ 725572  866585  772272 1233917  864786] int64
total net: 523162260


**Units:** KGS. For one record, net value is the money value of one order line: how many units were sold times the price of one unit (minus the tip flag). The total over all 811 records is about 523.2 million KGS. Vectorized code runs the loop inside compiled C code on a continuous typed array, so it is faster than a Python `for` loop (see 3.7).

### 3.4 Broadcasting

In [21]:
first6 = net[:6].reshape(6, 1)
factors = np.array([0.90, 1.00, 1.10])
scen = first6 * factors
print(first6.shape, factors.shape, scen.shape)
print(scen)

(6, 1) (3,) (6, 3)
[[ 653014.8  725572.   798129.2]
 [ 779926.5  866585.   953243.5]
 [ 695044.8  772272.   849499.2]
 [1110525.3 1233917.  1357308.7]
 [ 778307.4  864786.   951264.6]
 [1128991.5 1254435.  1379878.5]]


Shapes `(6, 1)` and `(3,)` are compatible: NumPy compares the shapes from the right. `3` against `1` works because a dimension of size 1 is stretched to 3. The missing leading dimension of `(3,)` is treated as 1, which is then stretched to 6. The result is `(6, 3)`. Each **row** is one record; each **column** is one scenario: 90%, 100% and 110% of the net value (the middle column equals the original value). With two arrays of the same shape, element `i` is multiplied only with element `i`; with broadcasting, every record is combined with every coefficient, and no copies are created.

### 3.5 Boolean indexing

In [22]:
m = (price_arr >= 3000) & ~(qty_arr > 25)
print("matches:", m.sum(), "of", len(m), "| total net:", net[m].sum(), "| share of all net: %.2f%%" % (net[m].sum() / net.sum() * 100))

matches: 388 of 811 | total net: 134957910 | share of all net: 25.80%


**388 of 811 records** have `Unit Price` of at least 3000 KGS and a quantity of 25 or less. They bring 134,957,910 KGS, which is **25.8%** of the net value. For management: almost half of the orders are smaller (quantity up to 25) but give only about a quarter of the money, so large-quantity orders are the main source of revenue.

### 3.6 Statistics and axes

In [23]:
print("row means:", scen.mean(axis=1))
print("col std:", scen.std(axis=0))
print("var:", np.var(net), "std:", np.std(net), "range:", net.max() - net.min())

row means: [ 725572.  866585.  772272. 1233917.  864786. 1254435.]
col std: [190711.09581617 211901.21757352 233091.33933087]
var: 309429813964.4606 std: 556264.1584395499 range: 2400220


`mean(axis=1)` collapses the columns and gives one mean per **row** (per record). Since the coefficients are 0.9, 1.0 and 1.1 (mean 1.0), each row mean equals the original net value. `std(axis=0)` collapses the rows and gives one standard deviation per **column** (per scenario). It grows with the coefficient (190,711 → 211,902 → 233,091), because scaling the values by k scales their standard deviation by k.

For the whole net array (all in KGS, so units are not mixed): variance ≈ 3.09 × 10¹¹ (KGS²), standard deviation ≈ 556,264 KGS, range = 2,400,220 KGS. **Interpretation:** a typical order differs from the average by about 556 thousand KGS, and the largest order is 2.4 million KGS bigger than the smallest, so order values are very spread out.

### 3.7 Performance optimization

Vectorized NumPy operations are faster than Python `for` loops because: (1) the loop is executed in precompiled C code, not by the Python interpreter, which would check types and create an object on every step; (2) an array stores values of one fixed type in a continuous block of memory, so the CPU reads it quickly and can use SIMD instructions; (3) there is no per-element function-call overhead. For arrays like these (811 elements) the difference is small, but it grows with data size.

## TASK 4. Business-Oriented Data Analysis

### 4.1 Reshape to long

`groupby().agg()` gives the totals; `pivot` makes the Branch × Meal Type table and `melt()` turns it into a long table with one combination per row.

In [24]:
d = df.copy()
g = d.groupby(["branch", "meal_type"]).agg(total=("unit_price", "sum")).reset_index()
tbl = g.pivot(index="branch", columns="meal_type", values="total")
print(tbl)
long = tbl.reset_index().melt(id_vars="branch", var_name="meal_type", value_name="total_unit_price")
print(long.sort_values("total_unit_price", ascending=False).head(5))

meal_type       Basic    Premium   Standard   Unknown
branch                                               
Bishkek     1449016.0  1010495.0  1312989.0   39970.0
Jalal-Abad  1678544.0  1684091.0  1256420.0   90863.0
Karakol     1045652.0  1232296.0  1124069.0  161040.0
Osh         1309870.0  1613305.0  1273063.0   53411.0
Tokmok      1269337.0  1573643.0  1479100.0       NaN
        branch meal_type  total_unit_price
6   Jalal-Abad   Premium         1684091.0
1   Jalal-Abad     Basic         1678544.0
8          Osh   Premium         1613305.0
9       Tokmok   Premium         1573643.0
14      Tokmok  Standard         1479100.0


**Strongest combination: Jalal-Abad × Premium, 1,684,091 KGS**, only slightly ahead of Jalal-Abad × Basic (1,678,544). Weakest known combination: Bishkek × Premium (1,010,495); `Unknown` rows are not a real category. Tokmok has no `Unknown` orders (NaN).

### 4.2 Pivot comparison

In [25]:
pt = pd.pivot_table(d, values="unit_price", index="meal_type", columns="payment_method", aggfunc="sum", fill_value=0)
print(pt)
print(pt.idxmax(axis=1))

payment_method    Basic  Premium  Standard
meal_type                                 
Basic           2060537  2261157   2430725
Premium         2289482  2522862   2301486
Standard        2384639  1709444   2351558
Unknown          135711   134096     75477
meal_type
Basic       Standard
Premium      Premium
Standard       Basic
Unknown        Basic
dtype: str


Best cell: **Premium meals paid with the Premium method, 2,522,862 KGS**. Weakest: Standard meals paid with the Premium method (1,709,444). Basic meals are strongest with the Standard method (2,430,725) and Standard meals with the Basic method (2,384,639). Premium meals are the only category whose best payment type is Premium; the other categories are spread more evenly, with 10-25% differences between payment types.

### 4.3 Descriptive statistics

In [26]:
print(d["unit_price"].describe())
print(d["unit_price"].agg(["sum", "mean", "median", "std"]))

count      811.000000
mean     25471.237978
std      14353.686964
min        151.000000
25%      13056.500000
50%      26329.000000
75%      38020.000000
max      49887.000000
Name: unit_price, dtype: float64
sum       2.065717e+07
mean      2.547124e+04
median    2.632900e+04
std       1.435369e+04
Name: unit_price, dtype: float64


Total = 20,657,174 KGS, mean = 25,471, median = 26,329, standard deviation = 14,354. The **median** better represents the typical record: it is not pulled by extreme values, which this dataset had before cleaning. After removing outliers the mean and median are close (the distribution is almost symmetric, skewness is −0.05), so both are fine now, but the median would stay stable if new outliers appeared. The standard deviation is more than half of the mean, so prices vary a lot between records.

### 4.4 Top and bottom

In [27]:
print(d.nlargest(3, "unit_price")[["order_id", "branch", "meal_type", "quantity", "unit_price"]])
print(d.nsmallest(3, "unit_price")[["order_id", "branch", "meal_type", "quantity", "unit_price"]])

      order_id   branch meal_type  quantity  unit_price
236  04_-00237  Bishkek  Standard        10       49887
201  04_-00202      Osh  Standard        42       49854
44   04_-00045   Tokmok  Standard         5       49851
      order_id      branch meal_type  quantity  unit_price
771  04_-00781     Bishkek     Basic        42         151
627  04_-00633  Jalal-Abad  Standard        13         265
601  04_-00607     Karakol   Premium        24         280


The top 3 prices (49,887, 49,854, 49,851) are all `Standard` meals and are almost identical, so there is a ceiling of about 50,000 KGS. The bottom 3 (151, 265, 280) are about **330 times** cheaper than the top. Prices as low as 151 KGS look like possible entry errors but are not flagged by the IQR rule (the lower bound is negative); they should be verified with the branches.

### 4.5 Ranking

`rank(method="average", ascending=False)` ranks every price inside its Branch (1 = highest); rows with rank ≤ 3 are the top 3 per Branch.

In [28]:
d["rank_in_branch"] = d.groupby("branch")["unit_price"].rank(method="average", ascending=False)
top3 = d[d["rank_in_branch"] <= 3].sort_values(["branch", "rank_in_branch"])
print(top3[["branch", "order_id", "unit_price", "rank_in_branch"]])

         branch   order_id  unit_price  rank_in_branch
236     Bishkek  04_-00237       49887             1.0
333     Bishkek  04_-00336       49184             2.0
741     Bishkek  04_-00751       48762             3.0
755  Jalal-Abad  04_-00765       49670             1.0
763  Jalal-Abad  04_-00773       49417             2.0
775  Jalal-Abad  04_-00785       49261             3.0
211     Karakol  04_-00212       49392             1.0
108     Karakol  04_-00109       48923             2.0
106     Karakol  04_-00107       48892             3.0
201         Osh  04_-00202       49854             1.0
764         Osh  04_-00774       49766             2.0
398         Osh  04_-00401       49666             3.0
44       Tokmok  04_-00045       49851             1.0
503      Tokmok  04_-00507       49435             2.0
18       Tokmok  04_-00019       49359             3.0


Each branch's top 3 prices are very close to each other (e.g. Osh: 49,854 / 49,766 / 49,666) and to the 50,000 ceiling, so the best individual orders are similar everywhere; branch differences come from the volume of orders, not from the top prices.